# DiffusionGemma — Google Colab (Free T4)

Runs `google/diffusiongemma-26B-A4B-it` on a free Colab T4 GPU using 4-bit quantization.

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(result.stdout or "No GPU found — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Takes ~2 min on first run
%pip install -q transformers accelerate bitsandbytes torch ipywidgets

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "google/diffusiongemma-26B-A4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")
print(f"Device map: {model.hf_device_map}")

## Basic generation

In [ ]:
def generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7) -> str:
    chat = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        chat, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


response = generate("Explain diffusion language models in 3 sentences.")
print(response)

## Multi-turn chat

In [ ]:
history = []

def chat(user_msg: str, max_new_tokens: int = 256) -> str:
    history.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        history, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)
    history.append({"role": "assistant", "content": reply})
    return reply


print(chat("What makes diffusion LLMs different from autoregressive ones?"))
print("---")
print(chat("Give a concrete example of where that difference matters."))

## Interactive playground

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_box = widgets.Textarea(
    value="Write a haiku about masked language modeling.",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="80px"),
)
max_tokens_slider = widgets.IntSlider(value=128, min=32, max=512, step=32, description="Max tokens")
temperature_slider = widgets.FloatSlider(value=0.7, min=0.0, max=1.5, step=0.05, description="Temperature")
run_button = widgets.Button(description="Generate", button_style="primary")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        print("Generating...")
        result = generate(prompt_box.value, max_tokens_slider.value, temperature_slider.value)
        clear_output()
        print(result)

run_button.on_click(on_click)
display(prompt_box, max_tokens_slider, temperature_slider, run_button, out)

## Batch comparison — same prompt, different temperatures

In [ ]:
prompt = "Give one surprising fact about language models."

for temp in [0.0, 0.5, 1.0, 1.4]:
    out_text = generate(prompt, max_new_tokens=80, temperature=max(temp, 1e-6))
    print(f"[temp={temp}] {out_text}")
    print()